# Member 5: Modeling & Evaluation

**Goal:** predict whether a movie earns ≥ $100M, and test whether critic reception adds predictive value.

**Input:** `data/final_dataset.csv` (Member 4) · **Code:** `src/modeling.py` · **Outputs:** `results/`

Pipeline: load → time-based split → 3 feature sets → 4 models (tuned) → metrics → plots → conclusions.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import pandas as pd
import modeling as M
pd.set_option("display.max_columns", 50)

## 1. Load data

In [ ]:
df = M.load_data()
print(df.shape)
df.head()

## 2. Target & time-based split
Train on movies before the split year, test on later movies (mimics predicting future releases).

In [ ]:
train_df, test_df = M.split_by_year(df)
summary = pd.DataFrame({
    "movies": [len(train_df), len(test_df)],
    "success rate": [train_df[M.TARGET].mean(), test_df[M.TARGET].mean()],
}, index=[f"train (< {M.SPLIT_YEAR})", f"test (>= {M.SPLIT_YEAR})"])
summary

The classes are imbalanced, so we use class weights and judge models by F1 and ROC-AUC rather than accuracy.

## 3. Feature sets
- **A** pre-release only · **B** A + critic scores · **C** B + post-release votes/popularity (leakage demo)

In [ ]:
sets = M.get_feature_sets(df)
for name, feats in sets.items():
    print(f"{name}: {len(feats)} features")

## 4. Train & tune all models
Each model is tuned with `RandomizedSearchCV` (5-fold stratified CV, ROC-AUC) on the training set only, then scored once on the test set.

In [ ]:
results, fitted = M.run_experiments(train_df, test_df, sets)
results

## 5. Do critic scores help? Is there leakage?

In [ ]:
results.pivot_table(index="Model", columns="Feature set", values="ROC-AUC")

In [ ]:
M.make_plots(results, fitted, test_df, show=True, save=False)

## 6. Best honest model: detailed report

In [ ]:
from sklearn.metrics import classification_report
fs, best_name = M.pick_best(results, fitted)
best_model, feats = fitted[(fs, best_name)]
print(f"Best model: {best_name} on {fs}")
print(classification_report(test_df[M.TARGET], best_model.predict(test_df[feats]),
                            target_names=["Flop", "Success"]))
M.get_importances(best_model, feats).head(10)

## 7. Save results for the report
This writes `results/metrics.csv`, `results/summary.json` and the figures. Then run `python src/make_report.py` from the repo root.

In [ ]:
M.save_outputs(df, train_df, test_df, sets, results, fitted)
print('Saved results/metrics.csv, results/summary.json and results/figures/')

## 8. Conclusions
Fill in after the run (or let `python src/make_report.py` generate them in `report/05_modeling_results.md`):
- Which model performed best and with what ROC-AUC / F1?
- Did critic scores improve ROC-AUC compared with Set A?
- How large is the leakage gap for Set C?
- Which features drive predictions (importance / SHAP)?